# [9.3] Emergent Misalignment Detection

> **Notebooks: [exercises](../../exercises/part3_emergent_misalignment_detection/9.3_Emergent_Misalignment_Detection_exercises.ipynb) | [solutions](../../exercises/part3_emergent_misalignment_detection/9.3_Emergent_Misalignment_Detection_solutions.ipynb)**

In [ ]:
GT_TIER = "GT-3"
EXERCISE_ID = "9_3_emergent_misalignment_detection"
EXPECTED_RUNTIME = "45-60 minutes for exercises; a few minutes for the pinned Pythia CUDA preflight"
REQUIRES_GPU = True

> **Local-first extension.** This section builds a safe proxy-drift detector.
> You will not train harmful models, generate harmful completions, or claim a
> reproduction of emergent misalignment. The real-model path uses pinned
> Pythia-70M hidden states and logits on benign behavior-policy descriptions.

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://info-arena.github.io/ARENA_img/slack.html), and ask any questions on the dedicated channels for this chapter of material.

If you want to change to dark mode, you can do this by clicking the three horizontal lines in the top-right, then navigating to Settings -> Theme.

Links to earlier chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/), [(3) LLM Evaluations](https://arena-chapter3-llm-evals.streamlit.app/), [(4) Alignment Science](https://arena-chapter4-alignment-science.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-14-1.png" width="350">


# Introduction

The phrase "emergent misalignment" should make you ask for unusually strong
evidence. A model can look worse after a change for many benign reasons: it may
be more sycophantic, more overconfident, weirdly format-obsessed, stylistically
off-task, or over-refusing safe requests. Those are not the same thing as a
dangerous finetuned model organism, but they are good places to practice the
measurement habits you would need before trusting a white-box safety claim.

In this section, the object you build is deliberately modest:

```text
a hidden-state detector for benign proxy drift, plus controls showing when not
to trust the detector.
```

The core question is:

```text
Can a simple white-box direction detect held-out benign proxy drift, align with
a behavior-logit proxy, fail the right negative controls, and support a bounded
mitigation check?
```

This is a preflight, not a broad safety benchmark. The CUDA report loads
`EleutherAI/pythia-70m-deduped`, extracts final hidden states and two next-token
logits (`" helpful"` and `" unsafe"`), trains a thresholded direction on safe
train contexts, and evaluates held-out contexts. It records hidden states and
logits only. It does not generate completions, train adapters, or reproduce
harmful emergent misalignment.

<img src="../assets/emergent_misalignment_validation_loop.svg" width="860">

The important habit is to demand all four pieces together:

```text
held-out detection works
behavior-logit alignment supports the feature interpretation
label-shuffled and random-direction controls fail
mitigation reduces the proxy score without moving neutral examples much
```


## Reading Material

For this notebook, you only need the framing from the original ARENA alignment
science material: black-box behavior is not enough when the claim is about an
internal mechanism. We use a safe proxy because the course should teach the
measurement pattern without asking you to create harmful models.

The report-backed result is best described as:

```text
Pythia-70M benign proxy-drift hidden-state preflight.
```

Do not describe it as:

```text
an emergent-misalignment reproduction;
a harmful finetuning result;
a trained crosscoder;
a broad unsafe-behavior detector;
a generated-completion evaluation.
```


## Content & Learning Objectives

### 1. Benign proxy drift taxonomy

You will keep the five safe proxy categories explicit and ordered.

> ##### Learning Objectives
>
> * Distinguish benign proxy drift from harmful model-organism work.
> * Preserve a stable taxonomy for later reports.
> * Avoid vague categories that make the claim impossible to audit.

### 2. Held-out drift detection

You will score detector logits against held-out drift labels.

> ##### Learning Objectives
>
> * Implement top-1 accuracy for binary detector logits.
> * Reject empty, non-finite, non-binary, or shape-invalid evidence.
> * Treat train and held-out examples as different evidence.

### 3. Proxy feature-score alignment

You will test whether white-box feature scores track behavior-logit deltas.

> ##### Learning Objectives
>
> * Use signed Pearson correlation, not absolute correlation.
> * Reject undefined correlations from constant inputs.
> * Keep feature evidence tied to behavior evidence.

### 4. Mitigation and early warning

You will bound a mitigation by drift reduction and capability preservation, then
compare white-box and black-box detection timing.

> ##### Learning Objectives
>
> * Require drift to go down without large neutral/capability movement.
> * Use strict timing for early-warning claims.
> * Separate the toy timing contract from the real-model hidden-state preflight.


## Setup code

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part3_emergent_misalignment_detection"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_emergent_misalignment_detection.tests as tests
import part3_emergent_misalignment_detection.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
ProxyDriftKind = Literal[
    "sycophantic",
    "overconfident",
    "json_only",
    "style_drift",
    "refusal_overgeneralizing",
]


@dataclass(frozen=True)
class DriftDetectorReport:
    detector_accuracy: float
    predicts_heldout_drift: bool


@dataclass(frozen=True)
class CrosscoderDriftAlignmentReport:
    correlation: float
    aligns_with_behavior_delta: bool


@dataclass(frozen=True)
class DriftMitigationReport:
    baseline_drift_score: float
    mitigated_drift_score: float
    drift_reduction: float
    capability_loss: float
    mitigation_passes: bool


@dataclass(frozen=True)
class EarlyWarningReport:
    white_box_detection_step: int
    black_box_detection_step: int
    white_box_catches_earlier: bool

In [ ]:
def _require_finite_tensor(name: str, tensor: t.Tensor) -> None:
    if tensor.numel() == 0:
        raise ValueError(f"{name} must be non-empty.")
    if not t.isfinite(tensor.float()).all():
        raise ValueError(f"{name} must contain only finite values.")


def _require_finite_scalar(name: str, value: float) -> None:
    value_tensor = t.tensor(value, dtype=t.float32)
    if not t.isfinite(value_tensor):
        raise ValueError(f"{name} must be finite.")


def _require_unit_interval(name: str, value: float) -> None:
    _require_finite_scalar(name, value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be between 0 and 1.")


def _require_correlation_threshold(name: str, value: float) -> None:
    _require_finite_scalar(name, value)
    if not -1.0 <= value <= 1.0:
        raise ValueError(f"{name} must be between -1 and 1.")


def _require_binary_detector_inputs(logits: t.Tensor, labels: t.Tensor) -> None:
    if logits.ndim != 2 or logits.shape[-1] != 2:
        raise ValueError("logits must have shape (batch, 2).")
    if labels.shape != (logits.shape[0],):
        raise ValueError("labels must have shape (batch,).")
    label_values = labels.long()
    if not labels.float().eq(label_values.float()).all():
        raise ValueError("labels must contain integer class ids.")
    if not label_values.ge(0).logical_and(label_values.le(1)).all():
        raise ValueError("labels must contain only 0 or 1.")

# Benign Proxy Drift Taxonomy

### Exercise 1 - define the benign proxy taxonomy

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend up to 5 mins on this exercise.
> ```

Return the exact benign proxy categories used throughout the section. The
ordering matters because the report and expected outputs use the same ordering.

In [ ]:
def safe_proxy_drift_kinds() -> tuple[ProxyDriftKind, ...]:
    raise NotImplementedError()


def proxy_kinds_smoke_test() -> list[str]:
    return list(safe_proxy_drift_kinds())


tests.test_proxy_kinds_are_explicit_safe_categories(proxy_kinds_smoke_test)

<details>
<summary>Expected output</summary>

```text
All tests in `test_proxy_kinds_are_explicit_safe_categories` passed!
```

</details>

<details>
<summary>Help - why keep proxy kinds explicit?</summary>

If the taxonomy is hidden in prose, the report can quietly change what it is
claiming to detect. A tuple is boring, but it forces the detector, mitigation,
controls, and final report to refer to the same benign phenomena.

</details>

<details>
<summary>Common bugs</summary>

* Returning a `set`, which loses stable ordering.
* Dropping `refusal_overgeneralizing`, which is one of the safety-relevant benign proxies.
* Adding unsafe or vague categories outside the artifact contract.

</details>

<details>
<summary>Solution</summary>

```python
def safe_proxy_drift_kinds() -> tuple[ProxyDriftKind, ...]:
    return (
        "sycophantic",
        "overconfident",
        "json_only",
        "style_drift",
        "refusal_overgeneralizing",
    )
```

</details>


# Held-Out Drift Detection

### Exercise 2 - score held-out proxy drift

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 10 mins on this exercise.
> ```

Implement binary detector accuracy. `logits[:, 0]` is the neutral score, and
`logits[:, 1]` is the proxy-drift score. Labels are held-out labels: `0` means
neutral and `1` means proxy drift.

In [ ]:
def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def drift_detector_report(
    detector_logits: t.Tensor,
    drift_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> DriftDetectorReport:
    raise NotImplementedError()


tests.test_drift_detector_report_scores_heldout_logits(drift_detector_report)

<details>
<summary>Expected output</summary>

```text
All tests in `test_drift_detector_report_scores_heldout_logits` passed!
```

The test includes a passing fixture, a flipped-logit failing fixture, and
invalid evidence cases such as empty batches, `nan` logits, one-class logits,
and out-of-range labels.

</details>

<details>
<summary>Help - what is held out here?</summary>

Held-out means the examples used for evaluation were not used to set the
direction or threshold. In the toy exercise this is represented by fixture
labels. In the CUDA report it means separate safe contexts: six train contexts
and four held-out contexts.

</details>

<details>
<summary>Common bugs</summary>

* Applying `softmax` and thresholding class-1 probability when argmax is enough.
* Allowing `(batch, 1)` logits, which can pass vacuously with all-zero labels.
* Treating train-context examples as held-out evidence.
* Letting `nan` or empty tensors silently produce meaningless accuracy.

</details>

<details>
<summary>Solution</summary>

```python
def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    _require_finite_tensor("logits", logits)
    _require_finite_tensor("labels", labels)
    _require_binary_detector_inputs(logits, labels)
    return float(logits.argmax(dim=-1).eq(labels.long()).float().mean().item())


def drift_detector_report(
    detector_logits: t.Tensor,
    drift_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> DriftDetectorReport:
    _require_unit_interval("min_accuracy", min_accuracy)
    accuracy = _prediction_accuracy(detector_logits, drift_labels)
    return DriftDetectorReport(
        detector_accuracy=accuracy,
        predicts_heldout_drift=accuracy >= min_accuracy,
    )
```

</details>

Now print a tiny report so you can see the fields that will later appear in the
verification artifact.

In [ ]:
def detector_smoke_test() -> dict:
    logits = t.tensor([[2.0, 0.0], [0.0, 2.0], [2.0, 0.0], [0.0, 2.0]])
    labels = t.tensor([0, 1, 0, 1])
    return drift_detector_report(logits, labels, min_accuracy=1.0).__dict__


utils.print_report("Drift detector", detector_smoke_test())
tests.test_detector_smoke_test(detector_smoke_test)

<details>
<summary>Expected output</summary>

```text
Drift detector:
  detector_accuracy: 1.0
  predicts_heldout_drift: True
All tests in `test_detector_smoke_test` passed!
```

</details>


# Proxy Feature-Score Alignment

### Exercise 3 - test whether feature scores track behavior deltas

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 10 mins on this exercise.
> ```

The real report compares the hidden-state detector score against a behavior
proxy derived from next-token logits. In this exercise, you implement the small
mathematical contract: signed Pearson correlation.

In [ ]:
def _pearson_correlation(left: t.Tensor, right: t.Tensor) -> float:
    raise NotImplementedError()


def crosscoder_drift_alignment_report(
    model_specific_feature_scores: t.Tensor,
    behavior_delta_scores: t.Tensor,
    *,
    min_correlation: float = 0.8,
) -> CrosscoderDriftAlignmentReport:
    raise NotImplementedError()


tests.test_crosscoder_alignment_uses_pearson_correlation(
    crosscoder_drift_alignment_report,
)

<details>
<summary>Expected output</summary>

```text
All tests in `test_crosscoder_alignment_uses_pearson_correlation` passed!
```

The visible test rejects anti-correlated evidence and constant-score evidence.
This matters because a detector can be confident for the wrong reason.

</details>

<details>
<summary>Help - why signed correlation?</summary>

Taking `abs(correlation)` would let an anti-feature pass. If high feature scores
predict lower drift behavior, then the feature is not aligned with the drift
interpretation. The sign is part of the claim.

</details>

<details>
<summary>Common bugs</summary>

* Comparing raw means rather than centered vectors.
* Taking absolute correlation, which incorrectly passes anti-correlated features.
* Returning `0.0` for constant inputs instead of rejecting undefined correlation.
* Ignoring shape mismatches between feature scores and behavior deltas.

</details>

<details>
<summary>Solution</summary>

```python
def _pearson_correlation(left: t.Tensor, right: t.Tensor) -> float:
    left = left.flatten().float()
    right = right.flatten().float()
    if left.shape != right.shape:
        raise ValueError("correlation inputs must have matching shape.")
    if left.numel() < 2:
        raise ValueError("at least two values are required for correlation.")
    _require_finite_tensor("left", left)
    _require_finite_tensor("right", right)
    left_centered = left - left.mean()
    right_centered = right - right.mean()
    denominator = left_centered.norm() * right_centered.norm()
    if denominator.item() == 0:
        raise ValueError("correlation is undefined for constant inputs.")
    return float((left_centered @ right_centered / denominator).item())


def crosscoder_drift_alignment_report(
    model_specific_feature_scores: t.Tensor,
    behavior_delta_scores: t.Tensor,
    *,
    min_correlation: float = 0.8,
) -> CrosscoderDriftAlignmentReport:
    _require_correlation_threshold("min_correlation", min_correlation)
    correlation = _pearson_correlation(
        model_specific_feature_scores,
        behavior_delta_scores,
    )
    return CrosscoderDriftAlignmentReport(
        correlation=correlation,
        aligns_with_behavior_delta=correlation >= min_correlation,
    )
```

</details>

In [ ]:
def crosscoder_smoke_test() -> dict:
    feature_scores = t.tensor([0.1, 0.8, 0.7, 0.2])
    behavior_delta = t.tensor([0.0, 0.9, 0.75, 0.1])
    return crosscoder_drift_alignment_report(
        feature_scores,
        behavior_delta,
        min_correlation=0.95,
    ).__dict__


utils.print_report("Proxy feature alignment", crosscoder_smoke_test())
tests.test_crosscoder_smoke_test(crosscoder_smoke_test)

<details>
<summary>Expected output</summary>

```text
Proxy feature alignment:
  correlation: 0.9994795322418213
  aligns_with_behavior_delta: True
All tests in `test_crosscoder_smoke_test` passed!
```

</details>


# Mitigation and Early Warning

### Exercise 4 - bound mitigation by drift and capability

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 10 mins on this exercise.
> ```

A mitigation claim should have two gates. The drift score should go down, and
ordinary neutral behavior should not move much. In the CUDA path, this is a
narrow LM-head projection check, not a behavioral deployment recipe.

In [ ]:
def drift_mitigation_report(
    baseline_drift_scores: t.Tensor,
    mitigated_drift_scores: t.Tensor,
    baseline_capability_scores: t.Tensor,
    mitigated_capability_scores: t.Tensor,
    *,
    min_drift_reduction: float = 0.2,
    max_capability_loss: float = 0.1,
) -> DriftMitigationReport:
    raise NotImplementedError()


tests.test_mitigation_report_bounds_capability_loss(drift_mitigation_report)

<details>
<summary>Expected output</summary>

```text
All tests in `test_mitigation_report_bounds_capability_loss` passed!
```

</details>

<details>
<summary>Help - what does mitigation need to preserve?</summary>

If a method reduces the drift score by destroying all useful behavior, it is not
a good mitigation. Here the "capability" score is a neutral proxy, so the test
only proves a narrow preservation check. It is still enough to teach the habit:
never report a safety intervention without measuring collateral damage.

</details>

<details>
<summary>Common bugs</summary>

* Checking drift reduction and forgetting capability loss.
* Using sums instead of means, which makes thresholds depend on fixture size.
* Treating negative capability loss as a failure; it means the proxy did not drop.
* Allowing empty or non-finite score tensors.

</details>

<details>
<summary>Solution</summary>

```python
def drift_mitigation_report(
    baseline_drift_scores: t.Tensor,
    mitigated_drift_scores: t.Tensor,
    baseline_capability_scores: t.Tensor,
    mitigated_capability_scores: t.Tensor,
    *,
    min_drift_reduction: float = 0.2,
    max_capability_loss: float = 0.1,
) -> DriftMitigationReport:
    if baseline_drift_scores.shape != mitigated_drift_scores.shape:
        raise ValueError("drift score tensors must match.")
    if baseline_capability_scores.shape != mitigated_capability_scores.shape:
        raise ValueError("capability score tensors must match.")
    _require_finite_tensor("baseline_drift_scores", baseline_drift_scores)
    _require_finite_tensor("mitigated_drift_scores", mitigated_drift_scores)
    _require_finite_tensor("baseline_capability_scores", baseline_capability_scores)
    _require_finite_tensor("mitigated_capability_scores", mitigated_capability_scores)
    _require_finite_scalar("min_drift_reduction", min_drift_reduction)
    _require_finite_scalar("max_capability_loss", max_capability_loss)
    baseline_drift = float(baseline_drift_scores.float().mean().item())
    mitigated_drift = float(mitigated_drift_scores.float().mean().item())
    drift_reduction = baseline_drift - mitigated_drift
    baseline_capability = float(baseline_capability_scores.float().mean().item())
    mitigated_capability = float(mitigated_capability_scores.float().mean().item())
    capability_loss = baseline_capability - mitigated_capability
    return DriftMitigationReport(
        baseline_drift_score=baseline_drift,
        mitigated_drift_score=mitigated_drift,
        drift_reduction=drift_reduction,
        capability_loss=capability_loss,
        mitigation_passes=(
            drift_reduction >= min_drift_reduction
            and capability_loss <= max_capability_loss
        ),
    )
```

</details>

In [ ]:
def mitigation_smoke_test() -> dict:
    baseline_drift = t.tensor([0.8, 0.7])
    mitigated_drift = t.tensor([0.3, 0.4])
    baseline_capability = t.tensor([0.9, 0.8])
    mitigated_capability = t.tensor([0.85, 0.78])
    return drift_mitigation_report(
        baseline_drift,
        mitigated_drift,
        baseline_capability,
        mitigated_capability,
        min_drift_reduction=0.3,
        max_capability_loss=0.1,
    ).__dict__


utils.print_report("Drift mitigation", mitigation_smoke_test())
tests.test_mitigation_smoke_test(mitigation_smoke_test)

<details>
<summary>Expected output</summary>

```text
Drift mitigation:
  baseline_drift_score: 0.75
  mitigated_drift_score: 0.35
  drift_reduction: 0.4
  capability_loss: 0.035
  mitigation_passes: True
All tests in `test_mitigation_smoke_test` passed!
```

</details>


### Exercise 5 - compare white-box and black-box detection timing

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend up to 5 mins on this exercise.
> ```

White-box monitoring is useful only if it catches something before a slower or
less granular black-box eval does. Equal-time detection is not early warning.

In [ ]:
def early_warning_report(
    *,
    white_box_detection_step: int,
    black_box_detection_step: int,
) -> EarlyWarningReport:
    raise NotImplementedError()


tests.test_early_warning_report_compares_detection_steps(early_warning_report)

<details>
<summary>Expected output</summary>

```text
All tests in `test_early_warning_report_compares_detection_steps` passed!
```

</details>

<details>
<summary>Help - why equal-time detection is not early warning</summary>

The word "early" is doing work. If a white-box detector fires at the same step
as the black-box eval, it may still be useful, but it is not an early-warning
signal. Use strict `<`, not `<=`.

</details>

<details>
<summary>Common bugs</summary>

* Using `<=`, which counts equal-time detection as earlier.
* Allowing negative detection steps.
* Reversing the white-box and black-box fields.

</details>

<details>
<summary>Solution</summary>

```python
def early_warning_report(
    *,
    white_box_detection_step: int,
    black_box_detection_step: int,
) -> EarlyWarningReport:
    if white_box_detection_step < 0 or black_box_detection_step < 0:
        raise ValueError("detection steps must be nonnegative.")
    return EarlyWarningReport(
        white_box_detection_step=white_box_detection_step,
        black_box_detection_step=black_box_detection_step,
        white_box_catches_earlier=white_box_detection_step < black_box_detection_step,
    )
```

</details>

In [ ]:
def early_warning_smoke_test() -> dict:
    return early_warning_report(
        white_box_detection_step=2,
        black_box_detection_step=5,
    ).__dict__


utils.print_report("Early warning", early_warning_smoke_test())
tests.test_early_warning_smoke_test(early_warning_smoke_test)

<details>
<summary>Expected output</summary>

```text
Early warning:
  white_box_detection_step: 2
  black_box_detection_step: 5
  white_box_catches_earlier: True
All tests in `test_early_warning_smoke_test` passed!
```

</details>


# Notebook Contract

### Exercise 6 - assemble the section smoke contract

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend up to 5 mins on this exercise.
> ```

The report runner calls `run_smoke_test`; keep this function small and
deterministic. It should aggregate the checks you just built, not run the CUDA
path.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "proxy_kinds": proxy_kinds_smoke_test(),
        "detector": detector_smoke_test(),
        "crosscoder": crosscoder_smoke_test(),
        "mitigation": mitigation_smoke_test(),
        "early_warning": early_warning_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Interpreting the smoke contract</summary>

This is not the real-model result. It is the small deterministic contract that
protects the learner implementation. The CUDA report below is the evidence for
the Pythia hidden-state preflight.

</details>

<details>
<summary>Common bugs</summary>

* Changing the top-level keys, which breaks the report runner.
* Returning dataclasses instead of dictionaries for the report summaries.
* Calling the GPU preflight from the smoke test.

</details>


# Signature Result

The CUDA path has one visible claim:

```text
On safe generated proxy-drift policy prompts, a Pythia-70M hidden-state
direction separates held-out benign drift prompts, aligns with a behavior-logit
proxy, fails the right controls, and supports a narrow LM-head projection
mitigation check.
```

<img src="../assets/emergent_misalignment_signature_result.svg" width="860">

| Quantity | Value |
| --- | ---: |
| Model | `EleutherAI/pythia-70m-deduped` |
| Revision | `e93a9faa9c77e5d09219f6c868bfc7a1bd65593c` |
| Hidden layer | `-1` |
| Train / held-out prompts | `36 / 24` |
| Train / held-out contexts | `6 / 4` |
| Drift kinds | `5` |
| Hidden state shape | `[24, 512]` |
| Held-out detector accuracy | `1.0` |
| Behavior proxy tokens | `" helpful"`, `" unsafe"` |
| Behavior alignment correlation | `0.7360556721687317` |
| Label-shuffled detector accuracy | `0.6666666865348816` |
| Random-direction detector accuracy | `0.375` |
| Black-box behavior proxy accuracy | `1.0` |
| Drift score margin | `4.652892112731934` |
| Mitigation drift-delta reduction | `1.0936462879180908` |
| Mitigation neutral-delta shift | `0.0` |
| Peak VRAM | `0.3098287582397461 GB` |
| Generated completions | `False` |

<details>
<summary>Interpreting the proxy-drift report</summary>

The detector accuracy says the hidden-state direction separates the held-out
safe proxy prompts in this small setup. The correlation says the hidden-state
score moves in the same direction as a next-token behavior proxy. The
label-shuffled and random-direction accuracies are deliberately not high enough
to pass. The mitigation number is a narrow readout check through the LM head,
not a deployment claim.

</details>

<details>
<summary>What would falsify this claim?</summary>

The claim should fail if the detector only works on train contexts, if a
label-shuffled direction passes, if a random direction passes, if behavior-logit
deltas are anti-correlated with the hidden-state score, if projection moves
neutral examples substantially, or if the report requires generated harmful
completions.

</details>


# CUDA Verification Path

The exercise notebook exposes the same GPU/full-experiment surface as the
section solution file. These wrappers call the real Pythia preflight; they are
not CPU fallbacks.

In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    from part3_emergent_misalignment_detection.solutions import (
        run_pythia_proxy_drift_preflight,
    )

    return run_pythia_proxy_drift_preflight(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)

The committed verification report is the evidence artifact for the real-model
path. This cell checks the report fields that matter for the scoped claim.

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "EleutherAI/pythia-70m-deduped"
assert gpu["hf_revision"] == "e93a9faa9c77e5d09219f6c868bfc7a1bd65593c"
assert gpu["hidden_layer"] == -1
assert gpu["generation_used"] is False
assert gpu["detector_accuracy"] == 1.0
assert gpu["predicts_heldout_drift"]
assert gpu["drift_alignment_correlation"] >= 0.7
assert gpu["aligns_with_behavior_delta"]
assert gpu["label_shuffled_detector_accuracy"] <= 0.75
assert gpu["random_direction_accuracy"] <= 0.55
assert gpu["black_box_behavior_proxy_accuracy"] == 1.0
assert gpu["mitigation_drift_delta_reduction"] >= 1.0
assert gpu["mitigation_neutral_delta_shift"] <= 0.1
assert gpu["mitigation_passes"]
assert gpu["heldout_prompt_count"] == 24
assert gpu["train_prompt_count"] == 36
assert gpu["drift_kind_count"] == 5
assert gpu["hidden_state_shape"] == [24, 512]
assert gpu["within_vram_budget"]

utils.print_report(
    "Pythia proxy-drift preflight",
    {
        "preflight_passed": gpu["preflight_passed"],
        "detector_accuracy": gpu["detector_accuracy"],
        "drift_alignment_correlation": gpu["drift_alignment_correlation"],
        "label_shuffled_detector_accuracy": gpu["label_shuffled_detector_accuracy"],
        "random_direction_accuracy": gpu["random_direction_accuracy"],
        "mitigation_drift_delta_reduction": gpu["mitigation_drift_delta_reduction"],
        "generation_used": gpu["generation_used"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    },
)
tests.test_committed_gpu_report_matches_proxy_drift_contract(report)

<details>
<summary>Expected output</summary>

```text
Pythia proxy-drift preflight:
  preflight_passed: True
  detector_accuracy: 1.0
  drift_alignment_correlation: 0.7360556721687317
  label_shuffled_detector_accuracy: 0.6666666865348816
  random_direction_accuracy: 0.375
  mitigation_drift_delta_reduction: 1.0936462879180908
  generation_used: False
  peak_vram_gb: 0.3098287582397461
All tests in `test_committed_gpu_report_matches_proxy_drift_contract` passed!
```

</details>

To regenerate this report locally:

```bash
BNB_CUDA_VERSION=130 uv run python scripts/run_extension_verification_reports.py --section 9.3 --max-vram-gb 24.0
```

Treat a failing report as a blocker. Do not weaken the page claims to make a
stale report look acceptable.


# Limitations

This section supports a narrow safe preflight, not a broad safety conclusion.

* The proxy drift categories are benign policy-description changes, not harmful finetunes.
* The real-model path uses hidden states and logits only; it does not evaluate generated completions.
* The "crosscoder-style" exercise tests feature-score alignment, but does not train a crosscoder.
* The toy early-warning timing check is a contract exercise; the Pythia report does not simulate a training trajectory.
* The mitigation check is an LM-head projection readout on hidden states, not a deployed intervention.
* The behavior proxy uses two next-token logits, so it is much narrower than a behavioral benchmark.


# Further Research

Good next steps would make the evidence broader without losing the controls:

* Replace the two-token behavior proxy with a small safe held-out behavior suite.
* Add a checkpoint-series setting where white-box detection timing is measured live.
* Train a real crosscoder or model-diffing feature map and compare it to the direction baseline.
* Stress-test prompt templates, contexts, and drift categories with bootstrap intervals.
* Compare projection mitigation to activation patching and causal ablation on the same safe task.